Naložimo knjižnico SageMath.
> Ubuntu + SageMath ~ 15GB prostora

> Docker Desktop (Ubuntu) + SageMath ~ 8GB prostora

In [2]:
from sage.all import *

Definiramo metriko razdalja, ki določi razdaljo med vozlišči v grafu. Razdalja = 
- 0 ... ko primerjamo vozlišče samega s seboj
- 1 ... ko primerjamo sosednji vozlišči
- 2 ... ko primerjamo nesosednji vozlišči

Naj bo:
- n ... število vozlišč,
- d ... število generatorjev.

Predpostavimo:
- n > 2d

In [3]:
def razdalja(w, u, n, d):
    if n <= 2 * d:
        raise ValueError("Pogoj n > 2d mora biti izpolnjen.")
    if w == u:
        return 0
    korak = (u - w) % n # (u-w)mod n ... ostanek pri deljenju z n
    # korak = i za i iz {1,..., n-d} predstavlja povezavo med 0. in i. vozliščem 
    return 1 if 1 <= korak <= n - d else 2  # za korake > n-d ni direktne povezave

Koda od Maše ___ . Prilagojena našemu problemu. Določamo metrično razdaljo, torej razdaljo med točko in množico točk.

In [4]:

def metricna_dimenzija_Cn(n, d, prelomi_simetrijo=True):
    """
    Izračuna metrično dimenzijo usmerjenega cirkulantnega grafa C_n(1, ..., n-d).
    Predpostavka: n > 2*d (zato so razdalje le 0, 1 ali 2).
    """
    if n <= 2 * d:
        raise ValueError("Pogoj n > 2d mora biti izpolnjen.")
    
    # Nastavitev linearnega programa (ILP)
    lp = MixedIntegerLinearProgram(maximization=False)
    x = lp.new_variable(binary=True) # izbere novo vozlišče iz grafa
    
    # Ciljna funkcija: minimiziramo število izbranih radarjev
    lp.set_objective(sum(x[w] for w in range(n)))
    
    # Lomljenje rotacijske simetrije (priporočeno v navodilih za hitrost pri večjih n)
    if prelomi_simetrijo:
        lp.add_constraint(x[0] == 1)

    # Pogoji: vsak par (u, v) mora biti ločen z vsaj enim radarjem
    for u in range(n):
        for v in range(u + 1, n):  # Gledamo samo u < v (brez podvajanja!)
            # Poiščemo vsa vozlišča w, ki ločijo par u in v
            locevalci = [w for w in range(n) if razdalja(w, u, n, d) != razdalja(w, v, n, d)]
            
            # Dodamo pogoj: vsaj eden izmed ločevalcev mora biti izbran
            lp.add_constraint(sum(x[w] for w in locevalci) >= 1)
            
    # Rešitev modela
    lp.solve()
    
    # Pridobimo izbrana vozlišča (pazimo na zaokroževanje pri float vrednostih)
    razresujoca_mnozica = [w for w in range(n) if round(lp.get_values(x[w])) == 1]
    dimenzija = len(razresujoca_mnozica)
    
    return dimenzija, razresujoca_mnozica

In [5]:
def je_razresujoca_mnozica(n, d, W):
    """Preverimo če je množica W razrešujoča. Torej da so vsi vektorji paroma različni."""
    if len(W) != len(set(W)) or any(w < 0 or w >= n for w in W):
        return False
    podpisi = [tuple(razdalja(w, v, n, d) for w in W) for v in range(n)]
    return len(set(podpisi)) == n

In [6]:
def test_znanih_formul(n_max=40):
    """Testiramo ILP za že poznane rezultate pri d=4 in d=5."""
    if n_max < 11:
        raise ValueError("n_max mora biti večji od 8 in 10.")

    stevilo_testov = 0
    for d, n_min in ((4, 10), (5, 11)):
        for n in range(n_min, n_max + 1):
            dimenzija, W = metricna_dimenzija_Cn(n, d)

            if len(W) != dimenzija:
                raise AssertionError(f"Moč množice in dmenzija sta različni pri n={n}, d={d}.")
            if not je_razresujoca_mnozica(n, d, W):
                raise AssertionError(f"Vrnjena množica ni razrešujoča za n={n}, d={d}.")

            if d == 4:
                expected = (2 * n + 4) // 5
            elif n % 3 == 0:
                expected = n // 3
            else:
                expected = n // 3 + 2

            if dimenzija != expected:
                raise AssertionError(
                    f"Pri n={n}, d={d}: dobimo {dimenzija}, pričakujemo pa {expected}."
                )
            stevilo_testov += 1

    print(f"Vseh {stevilo_testov} testov je bilo pravilnih.")

In [7]:
test_znanih_formul(n_max=40)

Vseh 61 testov je bilo pravilnih.


## Optimiziran solver za dolgi eksperiment

Ta razlicica uporabi HiGHS, Sageovo hitro metodo `lp.sum` (zaradi nižje časovne zahtevnosti napram navadnem sum) in rotacijsko simetrijo tudi pri pripravi omejitev. Ohrani isti vmesnik `metricna_dimenzija_Cn`, zato preostala koda ostane zdruzljiva.

In [8]:
def pripravi_model_Cn(
    n, d, prelomi_simetrijo=True, solver="HiGHS",
    casovna_omejitev=None, prikazi_dnevnik=False):
    """Pripravi ILP za C_n(1, ..., n-d), vendar ga se ne resi."""

    if n <= 2 * d:
        raise ValueError("Pogoj n > 2d mora biti izpolnjen.")

    lp = MixedIntegerLinearProgram(maximization=False, solver=solver)
    x = lp.new_variable(binary=True)

    # lp.sum je v Sage bistveno hitrejsi od Pythonovega sum za linearne izraze.
    lp.set_objective(lp.sum(x[w] for w in range(n)))

    if prelomi_simetrijo:
        lp.add_constraint(x[0] == 1)

    # Zaradi rotacijske simetrije locila za (u,v) dobimo z rotacijo locil
    # za (0,v-u). Razdalje zato izracunamo O(n^2)-krat namesto O(n^3)-krat.
    locila_po_razliki = {}
    for delta in range(1, n):
        locila_po_razliki[delta] = tuple(
            s for s in range(n)
            if razdalja(s, 0, n, d) != razdalja(s, delta, n, d))

    for u in range(n):
        for v in range(u + 1, n):
            delta = v - u
            lp.add_constraint(
                lp.sum(x[(u + s) % n] for s in locila_po_razliki[delta]) >= 1)

    if casovna_omejitev is not None:
        if str(solver).casefold() == "highs":
            lp.solver_parameter("time_limit", float(casovna_omejitev))
        else:
            lp.solver_parameter("timelimit", float(casovna_omejitev))

    if str(solver).casefold() == "highs":
        # Nicelna vrzel: sprejmemo samo dokazani optimum.
        lp.solver_parameter("mip_rel_gap", 0.0)
        lp.solver_parameter("mip_abs_gap", 0.0)
        lp.solver_parameter("log_to_console", bool(prikazi_dnevnik))

    return lp, x


def resi_pripravljeni_model(lp, x, n):
    """Resi pripravljen model in vrne dimenzijo ter optimalno mnozico."""
    lp.solve()
    vrednosti = lp.get_values(x, convert=ZZ, tolerance=1e-6)
    W = [w for w in range(n) if vrednosti[w] == 1]
    return len(W), W


def metricna_dimenzija_Cn(
    n, d, prelomi_simetrijo=True, solver="HiGHS",
    casovna_omejitev=None, prikazi_dnevnik=False):
    """Izracuna tocno metricno dimenzijo in optimalno razresujoco mnozico."""

    lp, x = pripravi_model_Cn(
        n, d, prelomi_simetrijo=prelomi_simetrijo, solver=solver,
        casovna_omejitev=casovna_omejitev, prikazi_dnevnik=prikazi_dnevnik)
    return resi_pripravljeni_model(lp, x, n)

## Eksperiment za $4 \leq d \leq 15$

Rezultat vsake instance se takoj doda v CSV, zato lahko prekinjen poskus pozneje nadaljujemo. Primeri so urejeni po narascajocem $n$, da pred tezjimi primeri dobimo cim vec rezultatov za vse vrednosti $d$. Casovna omejitev velja za vsak ILP posebej.

In [9]:
import csv
import json
from datetime import datetime
from pathlib import Path
from time import perf_counter
from sage.numerical.mip import MIPSolverException

STOLPCI_REZULTATOV = [
    "timestamp", "n", "d", "dimension", "resolving_set",
    "status", "certificate_ok", "runtime_seconds",
    "build_seconds", "solve_seconds", "solver",
    "timeout_seconds", "symmetry_break", "error"]


def _zadnji_rezultati(pot):
    """Vrne zadnjo zapisano vrstico za vsak par (n,d)."""
    pot = Path(pot)
    zadnji = {}
    if not pot.exists():
        return zadnji
    with pot.open("r", newline="", encoding="utf-8") as datoteka:
        for vrstica in csv.DictReader(datoteka):
            zadnji[(int(vrstica["n"]), int(vrstica["d"]))] = vrstica
    return zadnji


def _dodaj_vrstico(pot, vrstica):
    """Doda en dokoncan rezultat v checkpoint CSV."""
    pot = Path(pot)
    pot.parent.mkdir(parents=True, exist_ok=True)
    zapisi_glavo = not pot.exists() or pot.stat().st_size == 0
    with pot.open("a", newline="", encoding="utf-8") as datoteka:
        pisec = csv.DictWriter(datoteka, fieldnames=STOLPCI_REZULTATOV)
        if zapisi_glavo:
            pisec.writeheader()
        pisec.writerow(vrstica)


def izracunaj_instanco(
    n, d, casovna_omejitev=300, solver="HiGHS",
    prelomi_simetrijo=True, prikazi_dnevnik=False):
    """Izracuna eno instanco in vrne vrstico za CSV."""

    zacetek = perf_counter()
    zacetek_resevanja = None
    vrstica = {
        "timestamp": datetime.now().astimezone().isoformat(timespec="seconds"),
        "n": int(n), "d": int(d), "dimension": "",
        "resolving_set": "", "status": "error",
        "certificate_ok": False, "runtime_seconds": "",
        "build_seconds": "", "solve_seconds": "",
        "solver": solver, "timeout_seconds": float(casovna_omejitev),
        "symmetry_break": bool(prelomi_simetrijo), "error": ""}

    try:
        t0 = perf_counter()
        lp, x = pripravi_model_Cn(
            n, d, prelomi_simetrijo=prelomi_simetrijo, solver=solver,
            casovna_omejitev=casovna_omejitev, prikazi_dnevnik=prikazi_dnevnik)
        vrstica["build_seconds"] = round(perf_counter() - t0, 6)

        t0 = perf_counter()
        zacetek_resevanja = t0
        dimenzija, W = resi_pripravljeni_model(lp, x, n)
        vrstica["solve_seconds"] = round(perf_counter() - t0, 6)

        certifikat = je_razresujoca_mnozica(n, d, W)
        vrstica.update({
            "dimension": int(dimenzija),
            "resolving_set": json.dumps([int(w) for w in W]),
            "status": "optimal" if certifikat else "invalid_certificate",
            "certificate_ok": bool(certifikat)})

    except MIPSolverException as napaka:
        sporocilo = str(napaka)
        vrstica["status"] = (
            "timeout" if "time limit" in sporocilo.casefold() else "solver_error")

        vrstica["error"] = sporocilo

    except KeyboardInterrupt:
        print("\nUporabnik je prekinil instanco; ta ni bila zapisana.", flush=True)
        raise

    except Exception as napaka:
        vrstica["status"] = "error"
        vrstica["error"] = f"{type(napaka).__name__}: {napaka}"
        
    finally:
        if zacetek_resevanja is not None and vrstica["solve_seconds"] == "":
            vrstica["solve_seconds"] = round(perf_counter() - zacetek_resevanja, 6)
        vrstica["runtime_seconds"] = round(perf_counter() - zacetek, 6)

    return vrstica

In [10]:
def izvedi_eksperiment(
    d_vrednosti=range(4, 16), n_max=100, cas_na_instanco=300,
    skupni_cas_ur=8, pot="results/metric_dimensions.csv",
    solver="HiGHS", prelomi_simetrijo=True, prikazi_dnevnik=False,
    ustavi_d_po_timeoutu=True, ponovi_statuse=()):
    """
    Resuje dopustne pare (n,d), sproti shranjuje in podpira nadaljevanje.

    Za ponovni poskus timeoutov uporabi ponovi_statuse=("timeout",).
    Nova vrstica se doda v CSV; pri analizi obdrzimo zadnjo za vsak (n,d).
    """

    d_vrednosti = sorted({int(d) for d in d_vrednosti})
    if not d_vrednosti or min(d_vrednosti) < 4 or max(d_vrednosti) > 15:
        raise ValueError("d_vrednosti morajo biti med 4 in 15.")
    if n_max <= 2 * min(d_vrednosti):
        raise ValueError("n_max je premajhen za izbrane vrednosti d.")
    if cas_na_instanco <= 0 or skupni_cas_ur <= 0:
        raise ValueError("Casovni omejitvi morata biti pozitivni.")

    # Po n: najprej resimo lahke primere za vse d, sele nato vecje.
    kandidati = sorted(
        (n, d)
        for d in d_vrednosti
        for n in range(2 * d + 1, int(n_max) + 1))

    zadnji = _zadnji_rezultati(pot)
    ponovi_statuse = set(ponovi_statuse)
    prejsnji_timeout_d = {
        d for (n, d), vrstica in zadnji.items()
        if vrstica["status"] == "timeout" and "timeout" not in ponovi_statuse}

    kandidati = [ par for par in kandidati
        if par not in zadnji or zadnji[par]["status"] in ponovi_statuse]

    zacetek_eksperimenta = perf_counter()
    skupna_omejitev = 3600.0 * float(skupni_cas_ur)
    blokirani_d = set(prejsnji_timeout_d) if ustavi_d_po_timeoutu else set()
    uspesni = timeouti = 0

    print(f"Nacrt: {len(kandidati)} se neizracunanih primerov, "
        f"d={d_vrednosti}, n<={n_max}, solver={solver}.", flush=True)
    print(f"Omejitev: {cas_na_instanco:.0f} s/primer, "
        f"{skupni_cas_ur:g} h skupaj. CSV: {Path(pot)}", flush=True)

    if blokirani_d:
        print(f"Zaradi prejsnjih timeoutov preskakujem d={sorted(blokirani_d)}.", flush=True)

    for zaporedna, (n, d) in enumerate(kandidati, start=1):
        if d in blokirani_d:
            continue

        preteklo = perf_counter() - zacetek_eksperimenta
        preostalo = skupna_omejitev - preteklo

        if preostalo < 1.0:
            print("STOP: dosezena je skupna casovna omejitev.", flush=True)
            break

        trenutni_timeout = min(float(cas_na_instanco), preostalo)
        ura = datetime.now().astimezone().strftime("%H:%M:%S")
        print(f"\n[{zaporedna}/{len(kandidati)} | {ura}] START d={d}, n={n}; "
            f"solver najvec {trenutni_timeout:.0f} s.", flush=True)

        vrstica = izracunaj_instanco(
            n, d, casovna_omejitev=trenutni_timeout, solver=solver,
            prelomi_simetrijo=prelomi_simetrijo, prikazi_dnevnik=prikazi_dnevnik)

        _dodaj_vrstico(pot, vrstica)

        status = vrstica["status"]
        cas = float(vrstica["runtime_seconds"])
        if status == "optimal":
            uspesni += 1
            print(f"KONCANO d={d}, n={n}: dim={vrstica['dimension']}, "
                f"cas={cas:.2f} s, certifikat=OK.", flush=True)
            
        elif status == "timeout":
            timeouti += 1
            print(f"TIMEOUT d={d}, n={n} po {cas:.2f} s.", flush=True)

            if ustavi_d_po_timeoutu:
                blokirani_d.add(d)
                print(f"Nadaljnji n za d={d} bodo v tem zagonu preskoceni.", flush=True)

        else:
            print(f"STOP zaradi statusa {status}: {vrstica['error']}", flush=True)
            break

    skupaj = perf_counter() - zacetek_eksperimenta
    print(
        f"\nEksperiment koncan: {uspesni} optimalnih, {timeouti} timeoutov, "
        f"{skupaj / 60:.1f} min. Rezultati: {Path(pot)}", flush=True)
    return preberi_rezultate(pot)


def preberi_rezultate(pot="results/metric_dimensions.csv"):
    """Prebere CSV brez dodatnih knjiznic in obdrzi zadnji (n,d)."""
    zadnji = _zadnji_rezultati(pot)
    tabela = []
    for vrstica in zadnji.values():
        vrstica = dict(vrstica)
        vrstica["n"] = int(vrstica["n"])
        vrstica["d"] = int(vrstica["d"])
        vrstica["dimension"] = (
            int(float(vrstica["dimension"])) if vrstica["dimension"] else None)
        for ime in ("runtime_seconds", "build_seconds", "solve_seconds"):
            vrstica[ime] = float(vrstica[ime]) if vrstica[ime] else None
        tabela.append(vrstica)
    return sorted(tabela, key=lambda r: (r["d"], r["n"]))


def prikazi_povzetek(tabela, sirina_bloka=15):
    """Prikaze zadnje rezultate ter matriki dimenzij in casov."""
    from collections import Counter
    from html import escape
    from IPython.display import HTML, display

    def prikazi_html(vrstice):
        """Prikaze navadne HTML celice brez Sage/LaTeX ovojev \\( ... \\)."""
        glava, *podatki = vrstice
        deli = [
            '<div style="overflow-x:auto"><table style="border-collapse:collapse">',
            '<thead><tr>',
        ]
        deli.extend(
            f'<th style="border:1px solid #888;padding:4px 8px">{escape(str(x))}</th>'
            for x in glava
        )
        deli.append('</tr></thead><tbody>')
        for vrstica in podatki:
            deli.append('<tr>')
            deli.extend(
                f'<td style="border:1px solid #888;padding:4px 8px">'
                f'{escape("" if x is None else str(x))}</td>'
                for x in vrstica
            )
            deli.append('</tr>')
        deli.append('</tbody></table></div>')
        display(HTML(''.join(deli)))

    if not tabela:
        print("Tabela rezultatov je prazna.")
        return None, None

    stolpci = ["n", "d", "dimension", "status",
               "runtime_seconds", "build_seconds", "solve_seconds"]
    zadnjih_20 = [[r.get(ime, "") for ime in stolpci] for r in tabela[-20:]]
    print("Zadnjih 20 rezultatov:")
    prikazi_html([stolpci] + zadnjih_20)

    stevci = Counter(r["status"] for r in tabela)
    print("\nStevilo rezultatov po statusu:")
    prikazi_html([["status", "stevilo"]] + sorted(stevci.items()))

    optimalni = [r for r in tabela if r["status"] == "optimal"]
    if not optimalni:
        print("Ni optimalnih rezultatov za matriki.")
        return None, None

    matrika_dimenzij = {(r["d"], r["n"]): r["dimension"] for r in optimalni}
    matrika_casov = {(r["d"], r["n"]): r["runtime_seconds"] for r in optimalni}
    d_ji = sorted({r["d"] for r in optimalni})
    n_ji = sorted({r["n"] for r in optimalni})

    def izpisi_matriko(vrednosti, naslov, zaokrozi=False):
        print(f"\n{naslov}")
        for zacetek in range(0, len(n_ji), sirina_bloka):
            stolpci_n = n_ji[zacetek:zacetek + sirina_bloka]
            vrstice = [["d \\ n"] + stolpci_n]
            for d in d_ji:
                vrednosti_d = []
                for n in stolpci_n:
                    vrednost = vrednosti.get((d, n), "")
                    if zaokrozi and vrednost != "":
                        vrednost = round(vrednost, 3)
                    vrednosti_d.append(vrednost)
                vrstice.append([d] + vrednosti_d)
            prikazi_html(vrstice)

    izpisi_matriko(matrika_dimenzij, "Matrika metricnih dimenzij:")
    izpisi_matriko(matrika_casov, "Matrika celotnih casov v sekundah:", True)
    return matrika_dimenzij, matrika_casov

### Kratek preizkus (neobvezno)

Ta celica resi samo tri majhne primere in pise v loceno datoteko. Uporabi jo pred dolgim zagonom.

In [11]:
testna_tabela = izvedi_eksperiment(
    d_vrednosti=[4], n_max=11, cas_na_instanco=30, skupni_cas_ur=0.05,
    pot="results/test_metric_dimensions.csv", prikazi_dnevnik=False)
    
prikazi_povzetek(testna_tabela)

Nacrt: 0 se neizracunanih primerov, d=[4], n<=11, solver=HiGHS.
Omejitev: 30 s/primer, 0.0500000000000000 h skupaj. CSV: results/test_metric_dimensions.csv

Eksperiment koncan: 0 optimalnih, 0 timeoutov, 0.0 min. Rezultati: results/test_metric_dimensions.csv
Zadnjih 20 rezultatov:


n,d,dimension,status,runtime_seconds,build_seconds,solve_seconds
9,4,4,optimal,0.20694,0.096468,0.10659
10,4,4,optimal,0.047439,0.012148,0.034439
11,4,5,optimal,0.052463,0.020659,0.031502



Stevilo rezultatov po statusu:


status,stevilo
optimal,3



Matrika metricnih dimenzij:


d \ n,9,10,11
4,4,4,5



Matrika celotnih casov v sekundah:


d \ n,9,10,11
4,0.207,0.047,0.052


({(4, 9): 4, (4, 10): 4, (4, 11): 5},
 {(4, 9): 0.20694, (4, 10): 0.047439, (4, 11): 0.052463})

### Glavni zagon

Privzeto ima vsak primer najvec 5 minut, celoten zagon pa 8 ur. `START` vedno izpise tudi najdaljsi mozni cas trenutnega primera. Za podrobni HiGHS dnevnik nastavi `PRIKAZI_DNEVNIK = True`; privzeto je izklopljen, da izhod skoraj tisoc primerov ne upocasni VS Code. Ponoven zagon nadaljuje iz obstojecga CSV.

In [12]:
D_VREDNOSTI = range(4, 16)
N_MAX = 100
CAS_NA_INSTANCO = 5 * 60       # 5 minut
SKUPNI_CAS_UR = 8              # spremeni po potrebi
PRIKAZI_DNEVNIK = False        # True le za podrobni HiGHS dnevnik
POT_REZULTATOV = "results/metric_dimensions.csv"

In [13]:
rezultati = izvedi_eksperiment(
    d_vrednosti=D_VREDNOSTI, n_max=N_MAX,
    cas_na_instanco=CAS_NA_INSTANCO, skupni_cas_ur=SKUPNI_CAS_UR,
    pot=POT_REZULTATOV, prikazi_dnevnik=PRIKAZI_DNEVNIK)
    
matrika_dimenzij, matrika_casov = prikazi_povzetek(rezultati)

Nacrt: 972 se neizracunanih primerov, d=[4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], n<=100, solver=HiGHS.
Omejitev: 300 s/primer, 8 h skupaj. CSV: results/metric_dimensions.csv

[1/972 | 01:03:18] START d=4, n=9; solver najvec 300 s.
KONCANO d=4, n=9: dim=4, cas=0.04 s, certifikat=OK.

[2/972 | 01:03:18] START d=4, n=10; solver najvec 300 s.
KONCANO d=4, n=10: dim=4, cas=0.04 s, certifikat=OK.

[3/972 | 01:03:18] START d=4, n=11; solver najvec 300 s.
KONCANO d=4, n=11: dim=5, cas=0.04 s, certifikat=OK.

[4/972 | 01:03:18] START d=5, n=11; solver najvec 300 s.
KONCANO d=5, n=11: dim=5, cas=0.03 s, certifikat=OK.

[5/972 | 01:03:18] START d=4, n=12; solver najvec 300 s.
KONCANO d=4, n=12: dim=5, cas=0.05 s, certifikat=OK.

[6/972 | 01:03:18] START d=5, n=12; solver najvec 300 s.
KONCANO d=5, n=12: dim=4, cas=0.05 s, certifikat=OK.

[7/972 | 01:03:18] START d=4, n=13; solver najvec 300 s.
KONCANO d=4, n=13: dim=6, cas=0.05 s, certifikat=OK.

[8/972 | 01:03:18] START d=5, n=13; solver najv

n,d,dimension,status,runtime_seconds,build_seconds,solve_seconds
78,15,30,optimal,26.058434,0.281903,25.772233
79,15,30,optimal,18.913258,0.406818,18.501391
80,15,30,optimal,11.549477,0.249128,11.296317
81,15,30,optimal,7.346105,0.283965,7.059265
82,15,31,optimal,16.504453,0.434379,16.066661
83,15,32,optimal,35.589049,0.328363,35.255858
84,15,30,optimal,2.237385,0.508362,1.722223
85,15,32,optimal,13.362863,0.304606,13.052185
86,15,33,optimal,30.409751,0.334331,30.071725
87,15,30,optimal,2.990004,1.249237,1.73663



Stevilo rezultatov po statusu:


status,stevilo
optimal,936
solver_error,1



Matrika metricnih dimenzij:


d \ n,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
4,4,4,5,5,6,6,6,7,7,8,8,8,9,9,10
5,,,5,4,6,6,5,7,7,6,8,8,7,9,9
6,,,,,6,6,6,7,7,7,8,8,9,8,10
7,,,,,,,6,6,7,7,8,8,8,9,9
8,,,,,,,,,7,6,8,8,7,9,10
9,,,,,,,,,,,8,8,9,9,10
10,,,,,,,,,,,,,9,8,9
11,,,,,,,,,,,,,,,10
12,,,,,,,,,,,,,,,
13,,,,,,,,,,,,,,,


... (8 outputs left out)

d \ n,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53
4,0.604,0.56,1.337,0.933,1.656,2.211,1.03,2.34,1.683,4.9,3.926,2.57,3.385,2.877,7.265
5,0.137,0.325,0.436,0.25,0.226,0.8,0.652,0.364,0.447,0.164,0.362,0.506,0.246,0.434,0.529
6,0.343,0.32,0.441,0.353,0.468,1.449,2.919,0.442,0.399,0.472,0.607,1.794,0.685,1.576,0.522
7,0.296,0.342,1.179,0.325,0.433,0.686,0.842,0.532,0.706,0.977,0.722,0.461,1.308,0.808,0.749
8,0.119,0.537,0.579,0.151,0.622,0.559,0.621,0.487,0.427,0.185,0.881,0.442,0.204,1.406,0.582
9,0.299,0.44,0.298,0.828,0.613,0.602,1.205,0.46,0.42,0.456,0.899,4.05,0.395,5.073,0.962
10,0.685,0.364,0.382,0.332,0.402,0.598,1.57,0.395,0.618,0.412,0.302,1.831,0.455,0.746,1.025
11,0.135,0.339,1.109,0.191,4.842,0.648,0.185,0.919,3.739,0.231,0.522,2.154,0.318,0.772,3.941
12,1.184,0.716,0.402,2.466,0.984,0.477,3.21,0.186,2.346,0.633,0.306,1.979,3.522,1.022,1.305
13,0.664,0.57,0.462,0.295,1.222,0.519,0.718,0.993,0.593,0.597,3.909,0.557,3.409,0.898,0.812


d \ n,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68
4,4.386,7.829,5.773,4.726,8.842,8.577,5.238,10.154,7.256,14.267,12.696,8.727,15.525,9.622,27.082
5,0.211,0.321,1.181,0.192,0.503,0.544,0.256,0.638,0.5,0.616,0.661,0.658,0.313,0.693,0.739
6,1.748,0.514,3.504,2.156,0.602,0.584,1.904,0.797,0.755,3.099,0.69,3.179,0.71,8.586,3.021
7,1.691,0.685,0.506,2.328,0.658,3.643,3.175,0.68,4.501,2.891,2.456,5.597,3.129,8.146,6.285
8,0.212,0.591,0.526,0.224,0.825,0.555,0.261,0.776,0.642,0.252,1.019,0.714,0.512,1.754,0.969
9,0.61,0.815,0.652,1.816,0.755,4.968,2.389,0.654,4.31,2.63,1.042,1.055,2.605,8.941,0.853
10,1.189,0.583,1.529,0.519,2.733,0.676,0.651,0.735,0.947,1.343,6.96,0.895,1.101,1.009,0.849
11,0.501,5.805,4.018,0.318,5.45,3.615,0.268,1.13,4.405,0.649,7.06,4.702,0.607,9.467,6.888
12,3.575,4.091,0.664,1.439,5.339,0.582,7.125,1.865,0.687,3.367,0.874,5.014,0.929,8.018,18.9
13,2.05,1.372,0.729,1.053,0.646,1.425,8.954,0.583,1.93,2.252,0.543,14.768,0.828,1.595,1.174


d \ n,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83
4,18.758,33.056,24.39,14.33,36.654,31.37,39.673,43.195,36.406,82.727,49.364,30.336,69.851,32.638,84.814
5,0.284,0.768,1.008,0.306,1.385,1.462,0.534,0.923,1.505,0.446,2.074,0.857,0.388,1.075,0.997
6,1.06,1.163,4.66,1.063,1.101,5.893,1.556,10.505,2.169,19.545,11.449,6.247,2.285,10.869,2.938
7,1.692,9.057,6.595,3.017,11.471,6.559,19.721,14.511,5.886,19.511,11.599,11.788,19.157,11.709,30.39
8,0.466,1.089,0.889,0.601,1.43,1.061,0.434,1.742,1.548,0.492,2.419,1.112,1.316,1.801,2.134
9,15.285,1.125,1.075,3.783,1.347,9.909,2.647,13.979,9.422,1.637,18.484,10.233,6.288,1.306,9.648
10,3.094,1.696,1.066,2.063,2.48,2.389,2.133,2.355,3.209,5.014,2.101,8.246,6.296,1.628,15.866
11,0.355,10.573,6.992,0.576,10.208,8.814,0.561,16.877,8.221,0.688,18.714,9.86,0.58,16.932,13.347
12,0.733,7.923,5.668,2.216,12.13,7.467,6.657,1.849,9.541,20.432,2.628,25.741,20.314,1.5,29.726
13,1.107,1.956,5.147,1.546,1.021,8.334,2.402,13.39,2.968,1.665,15.869,1.945,1.623,1.484,3.283


d \ n,84,85,86,87,88,89,90,91,92,93,94,95,96,97
4,76.003,72.472,114.937,57.086,170.333,114.46,162.816,173.536,99.272,256.663,158.769,146.503,174.446,253.292
5,0.471,1.955,1.679,0.9,2.008,1.501,0.647,3.163,1.105,0.533,2.626,2.382,0.662,2.132
6,1.801,13.333,2.229,20.288,1.541,28.09,20.907,15.531,2.252,15.339,8.72,3.9,21.005,14.103
7,24.265,21.925,38.572,24.069,31.772,29.451,23.183,70.27,36.511,25.36,73.881,42.638,41.07,64.492
8,0.633,2.065,2.823,1.983,3.012,1.922,0.687,2.771,2.025,0.812,4.284,3.018,0.794,2.502
9,30.992,1.57,45.819,3.361,2.216,13.105,4.023,28.58,1.751,40.633,29.746,2.921,37.552,30.044
10,10.409,2.112,12.825,8.618,4.874,11.776,11.085,31.08,17.38,5.389,31.015,19.922,2.702,25.253
11,0.693,22.229,13.099,0.984,24.324,16.521,0.878,26.351,14.627,1.024,32.483,14.143,1.698,37.472
12,2.179,2.165,11.699,2.574,24.657,1.998,37.766,72.696,3.154,118.031,24.794,2.954,38.093,32.061
13,2.55,29.873,2.167,1.764,10.868,2.756,36.053,3.25,2.209,3.274,4.144,2.65,38.535,3.538
